In [45]:
from pathlib import Path
import numpy as np
import json
import xarray as xr

In [39]:
PROJECT_DIR = Path("..")

CURRENT_DATA_DIR = Path("../data/raw/ocean_current")

ARTIFACT_DIR = Path("../ml_service/artifacts")

current_files = sorted(
    list(CURRENT_DATA_DIR.glob("*.nc"))
)

SEA_ICE_DATA_PATH = Path(
    "../data/raw/sea_ice"
)

In [10]:
sample_current_ds = xr.open_dataset(
    current_files[0]
)

print(sample_current_ds)

<xarray.Dataset>
Dimensions:    (time: 1096, depth: 1, latitude: 121, longitude: 360)
Coordinates:
  * time       (time) datetime64[ns] 2023-01-01 2023-01-02 ... 2025-12-31
  * depth      (depth) float32 0.494
  * latitude   (latitude) float32 -70.0 -69.92 -69.83 ... -60.17 -60.08 -60.0
  * longitude  (longitude) float32 60.0 60.08 60.17 60.25 ... 89.75 89.83 89.92
Data variables:
    uo         (time, depth, latitude, longitude) float32 ...
    vo         (time, depth, latitude, longitude) float32 ...
Attributes:
    Conventions:       CF-1.11
    title:             daily mean fields from Global Ocean Physics Analysis a...
    institution:       MERCATOR OCEAN
    source:            MERCATOR GLORYS12V1
    history:           2023/06/01 16:20:05 MERCATOR OCEAN Netcdf creation
    references:        http://www.mercator-ocean.fr
    comment:           CMEMS product
    subset:source:     ARCO data downloaded from the Marine Data Store using ...
    subset:productId:  GLOBAL_MULTIYEAR_PHY

In [11]:
surface_current = sample_current_ds.sel(
    depth=0.494,
    method="nearest"
)

print(surface_current)

print("\nDepth selected:")
print(
    float(surface_current.depth.values)
)

print("\nuo shape:")
print(
    surface_current["uo"].shape
)

print("\nvo shape:")
print(
    surface_current["vo"].shape
)

<xarray.Dataset>
Dimensions:    (time: 1096, latitude: 121, longitude: 360)
Coordinates:
  * time       (time) datetime64[ns] 2023-01-01 2023-01-02 ... 2025-12-31
    depth      float32 0.494
  * latitude   (latitude) float32 -70.0 -69.92 -69.83 ... -60.17 -60.08 -60.0
  * longitude  (longitude) float32 60.0 60.08 60.17 60.25 ... 89.75 89.83 89.92
Data variables:
    uo         (time, latitude, longitude) float32 ...
    vo         (time, latitude, longitude) float32 ...
Attributes:
    Conventions:       CF-1.11
    title:             daily mean fields from Global Ocean Physics Analysis a...
    institution:       MERCATOR OCEAN
    source:            MERCATOR GLORYS12V1
    history:           2023/06/01 16:20:05 MERCATOR OCEAN Netcdf creation
    references:        http://www.mercator-ocean.fr
    comment:           CMEMS product
    subset:source:     ARCO data downloaded from the Marine Data Store using ...
    subset:productId:  GLOBAL_MULTIYEAR_PHY_001_030
    subset:datasetId:  

In [14]:
# Inspect ocean current data quality

uo = surface_current["uo"].values
vo = surface_current["vo"].values


print("uo")
print("Shape:", uo.shape)
print("NaN values:", np.isnan(uo).sum())
print("Minimum:", np.nanmin(uo))
print("Maximum:", np.nanmax(uo))


print("\nvo")
print("Shape:", vo.shape)
print("NaN values:", np.isnan(vo).sum())
print("Minimum:", np.nanmin(vo))
print("Maximum:", np.nanmax(vo))

uo
Shape: (1096, 121, 360)
NaN values: 10787928
Minimum: -1.1188085
Maximum: 1.2237922

vo
Shape: (1096, 121, 360)
NaN values: 10787928
Minimum: -1.3702811
Maximum: 1.1951048


In [15]:
# Calculate ocean current speed

current_speed = np.sqrt(
    uo ** 2 +
    vo ** 2
)


print("CURRENT SPEED")

print("Shape:", current_speed.shape)

print("Minimum speed:")
print(np.nanmin(current_speed))

print("Maximum speed:")
print(np.nanmax(current_speed))

print("Mean speed:")
print(np.nanmean(current_speed))

CURRENT SPEED
Shape: (1096, 121, 360)
Minimum speed:
0.0
Maximum speed:
1.4542248
Mean speed:
0.11769173


In [16]:
# Create ocean current validity mask
ocean_valid_mask = (
    ~np.isnan(uo)
    &
    ~np.isnan(vo)
)


print("Ocean validity mask shape:")
print(ocean_valid_mask.shape)

print("\nTotal values:")
print(ocean_valid_mask.size)

print("\nValid ocean values:")
print(np.sum(ocean_valid_mask))

print("\nMissing / invalid values:")
print(np.sum(~ocean_valid_mask))

print("\nValid percentage:")
print(
    np.sum(ocean_valid_mask)
    / ocean_valid_mask.size
    * 100
)

Ocean validity mask shape:
(1096, 121, 360)

Total values:
47741760

Valid ocean values:
36953832

Missing / invalid values:
10787928

Valid percentage:
77.40358126721763


In [17]:
# Check number of valid current cells per day

valid_current_cells_per_day = (
    ocean_valid_mask
    .reshape(
        ocean_valid_mask.shape[0],
        -1
    )
    .sum(axis=1)
)


print("Minimum valid cells in one day:")
print(valid_current_cells_per_day.min())

print("\nMaximum valid cells in one day:")
print(valid_current_cells_per_day.max())

print("\nUnique valid-cell counts:")
print(
    np.unique(
        valid_current_cells_per_day
    )
)

Minimum valid cells in one day:
33717

Maximum valid cells in one day:
33717

Unique valid-cell counts:
[33717]


In [18]:
# Create static ocean mask

ocean_spatial_mask = ocean_valid_mask[0]

print("Ocean spatial mask shape:")
print(ocean_spatial_mask.shape)

print("\nValid ocean cells:")
print(np.sum(ocean_spatial_mask))

print("\nInvalid cells:")
print(np.sum(~ocean_spatial_mask))

print("\nValid percentage:")
print(
    np.mean(ocean_spatial_mask) * 100
)

Ocean spatial mask shape:
(121, 360)

Valid ocean cells:
33717

Invalid cells:
9843

Valid percentage:
77.40358126721763


In [21]:
# Load sea-ice model geographic grids

lat_model_grid = np.load(
    ARTIFACT_DIR / "latitude_grid.npy"
)

lon_model_grid = np.load(
    ARTIFACT_DIR / "longitude_grid.npy"
)


print("Latitude grid shape:")
print(lat_model_grid.shape)

print("\nLongitude grid shape:")
print(lon_model_grid.shape)

print("\nLatitude range:")
print(
    np.nanmin(lat_model_grid),
    "to",
    np.nanmax(lat_model_grid)
)

print("\nLongitude range:")
print(
    np.nanmin(lon_model_grid),
    "to",
    np.nanmax(lon_model_grid)
)

Latitude grid shape:
(66, 57)

Longitude grid shape:
(66, 57)

Latitude range:
-72.47751 to -56.81248

Longitude range:
49.429558 to 89.78379


In [22]:
# Interpolate one day of currents

sample_day = 0


uo_sample = surface_current["uo"].isel(
    time=sample_day
)

vo_sample = surface_current["vo"].isel(
    time=sample_day
)


# Interpolate onto the 66 x 57 sea-ice grid

uo_on_model_grid = uo_sample.interp(
    latitude=(
        ("y", "x"),
        lat_model_grid
    ),
    longitude=(
        ("y", "x"),
        lon_model_grid
    ),
    method="linear"
)


vo_on_model_grid = vo_sample.interp(
    latitude=(
        ("y", "x"),
        lat_model_grid
    ),
    longitude=(
        ("y", "x"),
        lon_model_grid
    ),
    method="linear"
)


print("uo interpolated shape:")
print(uo_on_model_grid.shape)

print(
    "\nValid uo cells:"
)
print(
    np.sum(
        ~np.isnan(
            uo_on_model_grid.values
        )
    )
)


print("\nvo interpolated shape:")
print(vo_on_model_grid.shape)

print(
    "\nValid vo cells:"
)
print(
    np.sum(
        ~np.isnan(
            vo_on_model_grid.values
        )
    )
)

uo interpolated shape:
(66, 57)

Valid uo cells:
2113

vo interpolated shape:
(66, 57)

Valid vo cells:
2113


In [27]:
#Create common spatial mask

# Sea-ice valid mask
sea_ice_valid_mask = np.load(
    ARTIFACT_DIR / "spatial_mask.npy"
).astype(bool)



# Ocean current valid mask
ocean_current_valid_mask = (
    ~np.isnan(uo_on_model_grid.values)
    &
    ~np.isnan(vo_on_model_grid.values)
)


# Common cells where both datasets contain valid data
common_navigation_mask = (
    sea_ice_valid_mask
    &
    ocean_current_valid_mask
)


print("Sea-ice valid cells:")
print(np.sum(sea_ice_valid_mask))

print("\nOcean current valid cells:")
print(np.sum(ocean_current_valid_mask))

print("\nCommon valid cells:")
print(np.sum(common_navigation_mask))

print("\nCells lost when combining:")
print(
    np.sum(sea_ice_valid_mask)
    -
    np.sum(common_navigation_mask)
)

print("\nCommon grid percentage:")
print(
    np.sum(common_navigation_mask)
    /
    sea_ice_valid_mask.size
    *
    100
)

Sea-ice valid cells:
2108

Ocean current valid cells:
2113

Common valid cells:
2087

Cells lost when combining:
21

Common grid percentage:
55.47581073896863


In [28]:
# Check interpolated ocean-current coverage

test_indices = [
    0,
    365,
    730,
    1095
]

for day_index in test_indices:

    uo_day = surface_current["uo"].isel(
        time=day_index
    )

    vo_day = surface_current["vo"].isel(
        time=day_index
    )

    uo_interp = uo_day.interp(
        latitude=(
            ("y", "x"),
            lat_model_grid
        ),
        longitude=(
            ("y", "x"),
            lon_model_grid
        ),
        method="linear"
    )

    vo_interp = vo_day.interp(
        latitude=(
            ("y", "x"),
            lat_model_grid
        ),
        longitude=(
            ("y", "x"),
            lon_model_grid
        ),
        method="linear"
    )

    valid_mask = (
        ~np.isnan(uo_interp.values)
        &
        ~np.isnan(vo_interp.values)
    )

    common_mask_day = (
        sea_ice_valid_mask
        &
        valid_mask
    )

    print(
        f"Day {day_index} | "
        f"Date: {str(surface_current.time.values[day_index])[:10]}"
    )

    print(
        "Ocean valid cells:",
        np.sum(valid_mask)
    )

    print(
        "Common cells:",
        np.sum(common_mask_day)
    )

    print("-" * 40)

Day 0 | Date: 2023-01-01
Ocean valid cells: 2113
Common cells: 2087
----------------------------------------
Day 365 | Date: 2024-01-01
Ocean valid cells: 2113
Common cells: 2087
----------------------------------------
Day 730 | Date: 2024-12-31
Ocean valid cells: 2113
Common cells: 2087
----------------------------------------
Day 1095 | Date: 2025-12-31
Ocean valid cells: 2113
Common cells: 2087
----------------------------------------


In [30]:
# Interpolate complete ocean-current dataset

uo_all_on_grid = surface_current["uo"].interp(
    latitude=(
        ("y", "x"),
        lat_model_grid
    ),
    longitude=(
        ("y", "x"),
        lon_model_grid
    ),
    method="linear"
)


vo_all_on_grid = surface_current["vo"].interp(
    latitude=(
        ("y", "x"),
        lat_model_grid
    ),
    longitude=(
        ("y", "x"),
        lon_model_grid
    ),
    method="linear"
)


print("uo shape:")
print(uo_all_on_grid.shape)

print("\nvo shape:")
print(vo_all_on_grid.shape)

uo shape:
(1096, 66, 57)

vo shape:
(1096, 66, 57)


In [31]:
# Apply common navigation mask

uo_navigation = uo_all_on_grid.values.astype(
    np.float32
)

vo_navigation = vo_all_on_grid.values.astype(
    np.float32
)


# Remove cells outside the common navigation region

uo_navigation[
    :,
    ~common_navigation_mask
] = np.nan


vo_navigation[
    :,
    ~common_navigation_mask
] = np.nan


print("uo navigation shape:")
print(uo_navigation.shape)

print("\nvo navigation shape:")
print(vo_navigation.shape)

print("\nValid uo cells on first day:")
print(
    np.sum(
        ~np.isnan(
            uo_navigation[0]
        )
    )
)

print("\nValid vo cells on first day:")
print(
    np.sum(
        ~np.isnan(
            vo_navigation[0]
        )
    )
)

uo navigation shape:
(1096, 66, 57)

vo navigation shape:
(1096, 66, 57)

Valid uo cells on first day:
2087

Valid vo cells on first day:
2087


In [32]:
#Calculate current speed
# on the common navigation grid

current_speed_navigation = np.sqrt(
    uo_navigation ** 2
    +
    vo_navigation ** 2
)


print("Current speed shape:")
print(
    current_speed_navigation.shape
)


print("\nValid values:")
print(
    np.sum(
        ~np.isnan(
            current_speed_navigation
        )
    )
)


print("\nMinimum speed:")
print(
    np.nanmin(
        current_speed_navigation
    )
)


print("\nMaximum speed:")
print(
    np.nanmax(
        current_speed_navigation
    )
)


print("\nMean speed:")
print(
    np.nanmean(
        current_speed_navigation
    )
)

Current speed shape:
(1096, 66, 57)

Valid values:
2287352

Minimum speed:
7.495618e-05

Maximum speed:
1.4410578

Mean speed:
0.117740735


In [33]:
# Calculate ocean current direction

current_direction_navigation = np.degrees(
    np.arctan2(
        vo_navigation,
        uo_navigation
    )
)


# Convert range from
# -180 to 180
# into
# 0 to 360

current_direction_navigation = (
    current_direction_navigation
    +
    360
) % 360


print("Current direction shape:")
print(
    current_direction_navigation.shape
)


print("\nMinimum direction:")
print(
    np.nanmin(
        current_direction_navigation
    )
)


print("\nMaximum direction:")
print(
    np.nanmax(
        current_direction_navigation
    )
)

Current direction shape:
(1096, 66, 57)

Minimum direction:
6.1035156e-05

Maximum direction:
359.99994


In [36]:
# Verify ocean current time sequence

ocean_current_dates = surface_current.time.values

expected_dates = np.arange(
    np.datetime64("2023-01-01"),
    np.datetime64("2026-01-01"),
    dtype="datetime64[D]"
)

print("Ocean current dates:")
print(len(ocean_current_dates))

print("\nExpected dates:")
print(len(expected_dates))

print("\nFirst date:")
print(ocean_current_dates[0])

print("\nLast date:")
print(ocean_current_dates[-1])

print("\nDates match expected daily sequence:")
print(
    np.array_equal(
        ocean_current_dates,
        expected_dates
    )
)

Ocean current dates:
1096

Expected dates:
1096

First date:
2023-01-01T00:00:00.000000000

Last date:
2025-12-31T00:00:00.000000000

Dates match expected daily sequence:
True


In [37]:
# Create aligned ocean current dataset

ocean_current_navigation = {
    
    "dates": ocean_current_dates,
    
    "uo": uo_navigation,
    
    "vo": vo_navigation,
    
    "speed": current_speed_navigation,
    
    "direction": current_direction_navigation,
    
    "navigation_mask": common_navigation_mask
}


print("Ocean Current Navigation Dataset")
print("=" * 45)

for key, value in ocean_current_navigation.items():
    
    if isinstance(value, np.ndarray):
        print(
            f"{key}: {value.shape}"
        )
    
    else:
        print(
            f"{key}: {type(value)}"
        )

Ocean Current Navigation Dataset
dates: (1096,)
uo: (1096, 66, 57)
vo: (1096, 66, 57)
speed: (1096, 66, 57)
direction: (1096, 66, 57)
navigation_mask: (66, 57)


In [38]:
# Final quality checks

print("uo valid values:")
print(
    np.sum(
        ~np.isnan(
            uo_navigation
        )
    )
)

print("\nvo valid values:")
print(
    np.sum(
        ~np.isnan(
            vo_navigation
        )
    )
)

print("\nSpeed valid values:")
print(
    np.sum(
        ~np.isnan(
            current_speed_navigation
        )
    )
)

print("\nDirection valid values:")
print(
    np.sum(
        ~np.isnan(
            current_direction_navigation
        )
    )
)


print("\nExpected valid cells per day:")
print(
    np.sum(
        common_navigation_mask
    )
)


print("\nExpected total valid values:")
print(
    len(ocean_current_dates)
    *
    np.sum(common_navigation_mask)
)

uo valid values:
2287352

vo valid values:
2287352

Speed valid values:
2287352

Direction valid values:
2287352

Expected valid cells per day:
2087

Expected total valid values:
2287352


In [40]:
# Create processed ocean current folder

PROCESSED_CURRENT_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "ocean_current"
)

PROCESSED_CURRENT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("Processed data directory:")
print(PROCESSED_CURRENT_DIR.resolve())

Processed data directory:
D:\Jupyter\antarctic-navigation-ai\data\processed\ocean_current


In [41]:
# Save processed ocean current data

np.save(
    PROCESSED_CURRENT_DIR / "uo.npy",
    uo_navigation
)

np.save(
    PROCESSED_CURRENT_DIR / "vo.npy",
    vo_navigation
)

np.save(
    PROCESSED_CURRENT_DIR / "speed.npy",
    current_speed_navigation
)

np.save(
    PROCESSED_CURRENT_DIR / "direction.npy",
    current_direction_navigation
)

np.save(
    PROCESSED_CURRENT_DIR / "dates.npy",
    ocean_current_dates
)

np.save(
    PROCESSED_CURRENT_DIR / "navigation_mask.npy",
    common_navigation_mask
)


print("All ocean current arrays saved successfully")

All ocean current arrays saved successfully


In [43]:
# 7.20 Load processed ocean current data

test_uo = np.load(
    PROCESSED_CURRENT_DIR / "uo.npy"
)

test_vo = np.load(
    PROCESSED_CURRENT_DIR / "vo.npy"
)

test_speed = np.load(
    PROCESSED_CURRENT_DIR / "speed.npy"
)

test_dates = np.load(
    PROCESSED_CURRENT_DIR / "dates.npy"
)

test_mask = np.load(
    PROCESSED_CURRENT_DIR / "navigation_mask.npy"
)


print("uo shape:")
print(test_uo.shape)

print("\nvo shape:")
print(test_vo.shape)

print("\nSpeed shape:")
print(test_speed.shape)

print("\nDates shape:")
print(test_dates.shape)

print("\nNavigation mask shape:")
print(test_mask.shape)

print("\nValid navigation cells:")
print(np.sum(test_mask))

uo shape:
(1096, 66, 57)

vo shape:
(1096, 66, 57)

Speed shape:
(1096, 66, 57)

Dates shape:
(1096,)

Navigation mask shape:
(66, 57)

Valid navigation cells:
2087
